# Laboratorio de reconstrucción forense — Solo Bueno S.A.

Este notebook correlaciona los insumos del incidente sin ejecutar el código malicioso.

## Objetivos

1. Inventariar y calcular hashes de la evidencia.
2. Analizar el correo que solicitó un nuevo JWT.
3. Decodificar el JWT adjunto sin validar ni reutilizar credenciales.
4. Analizar conexiones VPN, reconocimiento, SMB y tráfico de salida.
5. Revisar los eventos de `WIN-SERVICIOS`.
6. Inspeccionar estáticamente `puravida.js`.
7. Examinar los archivos CSV potencialmente afectados.
8. Analizar el PCAP y construir una línea de tiempo correlacionada.
9. Exportar resultados para el informe.

> **Seguridad:** el notebook únicamente lee los archivos. No ejecuta `puravida.js`, no reproduce solicitudes HTTP y no intenta explotar servicios.


## 0. Preparación

En MyBinder, seleccione **Kernel → Restart Kernel and Run All Cells**.  
Los archivos deben permanecer dentro de la carpeta `evidencia/`.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import re
from datetime import datetime
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scapy.all import PcapReader, IP, TCP, UDP, Raw

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 200)

BASE = Path.cwd()
EVIDENCIA = BASE / "evidencia"
RESULTADOS = BASE / "resultados"
RESULTADOS.mkdir(exist_ok=True)

required = [
    "firewall.log", "web_access.log", "WIN_SERVICIOS_events.log",
    "captura_red.pcap", "captura_red.txt", "correos.html",
    "vpn_tokens.html", "puravida.js", "clientes.csv",
    "creditos.csv", "seguros.csv", "entregas.csv"
]
missing = [name for name in required if not (EVIDENCIA / name).exists()]
if missing:
    raise FileNotFoundError(f"Faltan archivos en evidencia/: {missing}")

print("Entorno listo.")
print("Directorio:", BASE)


## 1. Inventario y preservación lógica de la evidencia

Los hashes SHA-256 ayudan a documentar que los archivos analizados no cambiaron durante el laboratorio.


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

inventory = []
for path in sorted(EVIDENCIA.iterdir()):
    if path.is_file():
        inventory.append({
            "archivo": path.name,
            "tamaño_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

df_inventory = pd.DataFrame(inventory)
display(df_inventory)
df_inventory.to_csv(RESULTADOS / "inventario_hashes.csv", index=False)


## 2. Correo de solicitud del JWT

El archivo HTML contiene un arreglo JavaScript llamado `DATA`. Se extrae de forma local y se buscan mensajes relacionados con JWT, VPN, alertas y sucursales.


In [ ]:
html_text = (EVIDENCIA / "correos.html").read_text(encoding="utf-8",errors="replace"
)

match = re.search(r"const\s+DATA\s*=\s*(\[.*?\]);\s*let\s+sortKey",html_text,
    flags=re.S
)

if not match:
    raise ValueError(
        "No se pudo localizar el arreglo DATA en correos.html"
    )

emails = json.loads(match.group(1))

df_emails = pd.DataFrame(emails)

df_emails["fecha_dt"] = pd.to_datetime(
    df_emails["fecha"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

keywords = r"JWT|VPN|firewall|tráfico|página|sitio web|pedidos"

email_hits = df_emails[
    df_emails["asunto"].str.contains(keywords,case=False,na=False,regex=True)
    |
    df_emails["cuerpo"].str.contains(keywords,case=False,na=False,regex=True)
].copy()

email_hits = email_hits.sort_values("fecha_dt")

display(
    email_hits[["fecha", "from", "nombre", "asunto", "cuerpo"]
    ]
)

email_hits.to_csv(
    RESULTADOS / "correos_relevantes.csv",
    index=False
)

### 2.1 Revisión específica del correo de solicitud

Se comparan el dominio del remitente, el contenido del token adjunto y el socio comercial mencionado.


In [ ]:
jwt_requests = df_emails[df_emails["asunto"].str.contains("JWT", case=False, na=False)].copy()

rows = []
for _, email in jwt_requests.iterrows():
    attachments = email.get("adjuntos", []) or []
    for att in attachments:
        rows.append({
            "fecha": email["fecha"],
            "remitente": email["from"],
            "asunto": email["asunto"],
            "adjunto": att.get("nombre"),
            "contenido": att.get("data", ""),
        })

df_jwt_mail = pd.DataFrame(rows)
display(df_jwt_mail[["fecha", "remitente", "asunto", "adjunto"]])


## 3. Decodificación segura del JWT

La decodificación de `header` y `payload` no comprueba que el token sea legítimo. Solo permite leer sus claims.


In [ ]:
def b64url_decode_text(value: str) -> str:
    padding = "=" * (-len(value) % 4)
    return base64.urlsafe_b64decode(value + padding).decode("utf-8", errors="replace")

decoded_tokens = []
for _, row in df_jwt_mail.iterrows():
    token = str(row["contenido"]).strip()
    parts = token.split(".")
    if len(parts) != 3:
        decoded_tokens.append({"error": "Formato JWT inválido", "token": token})
        continue
    try:
        header = json.loads(b64url_decode_text(parts[0]))
        payload = json.loads(b64url_decode_text(parts[1]))
        decoded_tokens.append({
            "correo_solicitante": row["remitente"],
            "algoritmo": header.get("alg"),
            "nombre_claim": payload.get("name"),
            "correo_claim": payload.get("email"),
            "scope": payload.get("scope"),
            "issuer": payload.get("iss"),
            "iat": pd.to_datetime(payload.get("iat"), unit="s", errors="coerce"),
            "exp": pd.to_datetime(payload.get("exp"), unit="s", errors="coerce"),
        })
    except Exception as exc:
        decoded_tokens.append({"error": str(exc)})

df_tokens = pd.DataFrame(decoded_tokens)
display(df_tokens)
df_tokens.to_csv(RESULTADOS / "jwt_decodificado.csv", index=False)


## 4. Análisis de la herramienta de JWT

Se buscan indicadores de riesgo en el código, como secretos embebidos y los claims utilizados. El secreto se presenta enmascarado en los resultados.


In [ ]:
vpn_tool = (EVIDENCIA / "vpn_tokens.html").read_text(encoding="utf-8", errors="replace")

secret_match = re.search(r'const\s+SECRET\s*=\s*"([^"]+)"', vpn_tool)
payload_match = re.search(r"const\s+payload\s*=\s*\{([^}]+)\}", vpn_tool)

jwt_tool_findings = {
    "secreto_embebido": bool(secret_match),
    "longitud_secreto": len(secret_match.group(1)) if secret_match else None,
    "secreto_mascarado": (
        secret_match.group(1)[:2] + "*" * max(0, len(secret_match.group(1)) - 4) + secret_match.group(1)[-2:]
        if secret_match else None
    ),
    "payload_encontrado": payload_match.group(1).strip() if payload_match else None,
}
display(pd.DataFrame([jwt_tool_findings]))


## 5. Bitácora del firewall

Se parsean los campos principales: fecha, acción, protocolo, origen, destino, puertos, enlace y datos adicionales.


In [ ]:
fw_pattern = re.compile(
    r"^(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s+"
    r"(?P<action>\S+)\s+(?P<proto>\S+)\s+"
    r"(?P<src>\S+)\s*:(?P<src_port>\S+)\s+->\s+"
    r"(?P<dst>\S+)\s*:(?P<dst_port>\S+)\s+"
    r"(?P<rest>.*)$"
)

fw_rows = []
for line in (EVIDENCIA / "firewall.log").read_text(encoding="utf-8", errors="replace").splitlines():
    m = fw_pattern.match(line)
    if not m:
        continue
    row = m.groupdict()
    row["timestamp"] = pd.to_datetime(row["timestamp"])
    for key in ["src_port", "dst_port"]:
        row[key] = pd.to_numeric(row[key], errors="coerce")
    extras = dict(re.findall(r"(\w[\w-]*)=([^\s]+)", row["rest"]))
    row.update(extras)
    row["raw"] = line
    fw_rows.append(row)

df_fw = pd.DataFrame(fw_rows).sort_values("timestamp")
print("Eventos parseados:", len(df_fw))
display(df_fw.head())


### 5.1 Conexión VPN sospechosa y reconocimiento


In [ ]:
vpn_or_scan = df_fw[
    df_fw["raw"].str.contains(r"10\.8\.0\.37|201\.203\.88\.47", regex=True, na=False)
].copy()

display(vpn_or_scan[["timestamp", "action", "proto", "src", "src_port", "dst", "dst_port", "raw"]])
vpn_or_scan.to_csv(RESULTADOS / "firewall_vpn_reconocimiento.csv", index=False)

scan_ports = (
    vpn_or_scan[vpn_or_scan["src"].eq("10.8.0.37")]
    .groupby(["dst_port", "action"], dropna=False)
    .size()
    .reset_index(name="intentos")
    .sort_values("dst_port")
)
display(scan_ports)


### 5.2 Tráfico potencialmente relacionado con exfiltración


In [ ]:
exfil_fw = df_fw[
    df_fw["raw"].str.contains(r"45\.77\.132\.90|elgarrotazo\.co|/collect", regex=True, na=False)
].copy()

display(exfil_fw[["timestamp", "src", "src_port", "dst", "dst_port", "bytes", "host", "uri", "method", "raw"]])
exfil_fw.to_csv(RESULTADOS / "firewall_exfiltracion.csv", index=False)

print("Inicio:", exfil_fw["timestamp"].min())
print("Fin:", exfil_fw["timestamp"].max())
print("Cantidad de registros:", len(exfil_fw))


> **Nota forense:** no se debe sumar automáticamente el campo `bytes` como volumen real exfiltrado sin comprender si representa incrementos, acumulados o valores simulados. El PCAP y los tamaños reales de los CSV deben usarse para contrastar esa cifra.


## 6. Eventos de Windows en `WIN-SERVICIOS`


In [ ]:
event_pattern = re.compile(
    r"^(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s+"
    r"EventID=(?P<event_id>\d+)\s+Source=(?P<source>.*?)\s{2,}(?P<message>.*)$"
)

event_rows = []
for line in (EVIDENCIA / "WIN_SERVICIOS_events.log").read_text(encoding="utf-8", errors="replace").splitlines():
    m = event_pattern.match(line)
    if m:
        row = m.groupdict()
        row["timestamp"] = pd.to_datetime(row["timestamp"])
        row["event_id"] = int(row["event_id"])
        row["raw"] = line
        event_rows.append(row)

df_events = pd.DataFrame(event_rows).sort_values("timestamp")
display(df_events)
df_events.to_csv(RESULTADOS / "eventos_windows_parseados.csv", index=False)


## 7. Análisis estático de `puravida.js`

No se ejecuta el script. Solo se extraen rutas, destino de red, método HTTP, encabezados y acción de eliminación.


In [ ]:
js_text = (EVIDENCIA / "puravida.js").read_text(encoding="utf-8", errors="replace")

shares = re.findall(r'"(\\\\\\\\[^"]+)"', js_text)
url_match = re.search(r'var\s+c2\s*=\s*"([^"]+)"', js_text)
method_match = re.search(r'xhr\.open\("([^"]+)"', js_text)
headers = re.findall(r'xhr\.setRequestHeader\("([^"]+)",\s*"([^"]+)"\)', js_text)
delete_match = re.search(r'fso\.DeleteFile\("([^"]+)"\)', js_text)

df_js = pd.DataFrame([{
    "archivos_objetivo": shares,
    "destino": url_match.group(1) if url_match else None,
    "metodo": method_match.group(1) if method_match else None,
    "encabezados": headers,
    "lee_archivo_completo": "ReadAll()" in js_text,
    "envia_datos": "xhr.send(data)" in js_text,
    "autoeliminacion": delete_match.group(1) if delete_match else None,
}])
display(df_js.T)
df_js.astype(str).to_csv(RESULTADOS / "analisis_puravida.csv", index=False)


## 8. Archivos CSV potencialmente afectados

Se revisan esquema, cantidad de registros, tamaño y duplicados por cédula. Los valores personales no se imprimen completos para minimizar exposición.


In [ ]:
csv_summary = []
csv_frames = {}

for name in ["creditos.csv", "clientes.csv", "seguros.csv", "entregas.csv"]:
    path = EVIDENCIA / name
    df = pd.read_csv(path)
    csv_frames[name] = df
    id_col = next((c for c in df.columns if "ced" in c.lower() or "céd" in c.lower()), None)
    csv_summary.append({
        "archivo": name,
        "registros": len(df),
        "columnas": len(df.columns),
        "tamaño_bytes": path.stat().st_size,
        "campos": ", ".join(map(str, df.columns)),
        "duplicados_identificador": int(df[id_col].duplicated().sum()) if id_col else None,
    })

df_csv_summary = pd.DataFrame(csv_summary)
display(df_csv_summary)
df_csv_summary.to_csv(RESULTADOS / "resumen_csv.csv", index=False)


## 9. Análisis del PCAP con Scapy

Se genera un resumen de conversaciones IP/TCP/UDP y se buscan los indicadores conocidos. Scapy permite procesar el PCAP directamente dentro de Binder sin interfaz gráfica.


In [ ]:
pcap_path = EVIDENCIA / "captura_red.pcap"
packet_rows = []
payload_hits = []

iocs = {
    "10.8.0.37",
    "10.10.4.10",
    "45.77.132.90",
    "181.193.94.95",
    "181.193.94.94",
}

with PcapReader(str(pcap_path)) as reader:
    for idx, pkt in enumerate(reader, start=1):
        if IP not in pkt:
            continue
        row = {
            "packet": idx,
            "timestamp_epoch": float(pkt.time),
            "src": pkt[IP].src,
            "dst": pkt[IP].dst,
            "protocol": "IP",
            "sport": None,
            "dport": None,
            "length": len(pkt),
        }
        if TCP in pkt:
            row.update({"protocol": "TCP", "sport": pkt[TCP].sport, "dport": pkt[TCP].dport})
        elif UDP in pkt:
            row.update({"protocol": "UDP", "sport": pkt[UDP].sport, "dport": pkt[UDP].dport})
        packet_rows.append(row)

        if Raw in pkt:
            raw = bytes(pkt[Raw].load)
            text = raw.decode("utf-8", errors="ignore")
            if any(term in text for term in ["elgarrotazo.co", "/collect", "POST ", "X-File"]):
                payload_hits.append({
                    **row,
                    "payload_preview": text[:500].replace("\\r", "\\\\r").replace("\\n", "\\\\n")
                })

df_packets = pd.DataFrame(packet_rows)
df_packets["timestamp"] = pd.to_datetime(df_packets["timestamp_epoch"], unit="s", errors="coerce")
print("Paquetes IP:", len(df_packets))
display(df_packets.head())


In [ ]:
ioc_packets = df_packets[df_packets["src"].isin(iocs) | df_packets["dst"].isin(iocs)].copy()
display(ioc_packets.head(100))
ioc_packets.to_csv(RESULTADOS / "pcap_paquetes_ioc.csv", index=False)

conversations = (
    df_packets.groupby(["src", "dst", "protocol", "sport", "dport"], dropna=False)
    .agg(paquetes=("packet", "count"), bytes_observados=("length", "sum"),
         inicio=("timestamp", "min"), fin=("timestamp", "max"))
    .reset_index()
    .sort_values(["bytes_observados", "paquetes"], ascending=False)
)
display(conversations.head(50))
conversations.to_csv(RESULTADOS / "pcap_conversaciones.csv", index=False)

df_payload_hits = pd.DataFrame(payload_hits)
display(df_payload_hits)
df_payload_hits.to_csv(RESULTADOS / "pcap_payloads_relevantes.csv", index=False)


### 9.1 Visualización de actividad por minuto


In [ ]:
suspicious_packets = df_packets[
    (df_packets["src"].isin({"10.8.0.37", "10.10.4.10"}))
    | (df_packets["dst"].isin({"10.10.4.10", "45.77.132.90"}))
].copy()

if not suspicious_packets.empty:
    by_minute = (
        suspicious_packets.set_index("timestamp")
        .resample("1min")
        .agg(paquetes=("packet", "count"), bytes=("length", "sum"))
    )
    display(by_minute[by_minute["paquetes"] > 0])

    ax = by_minute["paquetes"].plot(figsize=(12, 4), marker="o", title="Paquetes sospechosos por minuto")
    ax.set_xlabel("Hora")
    ax.set_ylabel("Cantidad de paquetes")
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron paquetes con los IOC configurados.")


## 10. Revisión de `captura_red.txt`

El texto permite buscar rápidamente términos concretos y contrastar resultados con el PCAP.


In [ ]:
capture_text_lines = (EVIDENCIA / "captura_red.txt").read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "10.8.0.37", "10.10.4.10", "45.77.132.90",
    "elgarrotazo.co", "/collect", "SMB1", "POST"
]

text_hits = []
for line_no, line in enumerate(capture_text_lines, start=1):
    matched = [term for term in terms if term.lower() in line.lower()]
    if matched:
        text_hits.append({
            "linea": line_no,
            "terminos": ", ".join(matched),
            "contenido": line,
        })

df_capture_hits = pd.DataFrame(text_hits)
display(df_capture_hits.head(300))
df_capture_hits.to_csv(RESULTADOS / "captura_txt_hallazgos.csv", index=False)


## 11. Bitácora web y afectación de disponibilidad

Se analiza el período alrededor de las alertas del enlace web y se cuentan códigos HTTP, especialmente `503`.


In [ ]:
web_pattern = re.compile(
    r'^(?P<src>\S+)\s+\S+\s+\S+\s+\[(?P<timestamp>[^\]]+)\]\s+'
    r'"(?P<method>\S+)\s+(?P<uri>\S+)\s+(?P<http_version>[^"]+)"\s+'
    r'(?P<status>\d+)\s+(?P<bytes>\d+|-)\s+"(?P<referer>[^"]*)"\s+"(?P<agent>[^"]*)"'
)

web_rows = []
for line in (EVIDENCIA / "web_access.log").read_text(encoding="utf-8", errors="replace").splitlines():
    m = web_pattern.match(line)
    if m:
        row = m.groupdict()
        row["timestamp"] = pd.to_datetime(
            row["timestamp"], format="%d/%b/%Y:%H:%M:%S %z", errors="coerce"
        )
        row["status"] = int(row["status"])
        row["bytes"] = pd.to_numeric(row["bytes"], errors="coerce")
        web_rows.append(row)

df_web = pd.DataFrame(web_rows)
print("Solicitudes web parseadas:", len(df_web))
display(df_web.head())


In [ ]:
if not df_web.empty:
    status_counts = df_web["status"].value_counts().rename_axis("status").reset_index(name="cantidad")
    display(status_counts)

    web_503 = df_web[df_web["status"].eq(503)].copy()
    print("Respuestas 503:", len(web_503))
    display(web_503.head(100))
    web_503.to_csv(RESULTADOS / "web_respuestas_503.csv", index=False)

    top_sources = (
        df_web.groupby("src")
        .agg(solicitudes=("src", "size"), bytes=("bytes", "sum"))
        .reset_index()
        .sort_values("solicitudes", ascending=False)
    )
    display(top_sources.head(30))
    top_sources.to_csv(RESULTADOS / "web_fuentes_principales.csv", index=False)


## 12. Construcción de la línea de tiempo correlacionada

La línea de tiempo combina eventos objetivos de correos, firewall, Windows y artefactos. Las interpretaciones deben distinguirse de los hechos observados.


In [ ]:
timeline = []

# Email JWT
for _, row in jwt_requests.iterrows():
    timeline.append({
        "timestamp": row["fecha_dt"],
        "fuente": "correos.html",
        "tipo": "Solicitud de acceso",
        "hecho_observado": f"Correo de {row['from']}: {row['asunto']}",
        "interpretacion": "Solicitud de reemisión de JWT que requiere validación de identidad.",
        "certeza": "Alta (correo observado)",
    })

# Relevant firewall events
for _, row in df_fw[
    df_fw["raw"].str.contains(
        r"vpn-tunnel established|10\.8\.0\.37|45\.77\.132\.90|THRESHOLD|EVENT END",
        regex=True, na=False
    )
].iterrows():
    if row["action"] == "DROP" and row["src"] == "10.8.0.37":
        tipo = "Reconocimiento"
        interpretation = "Intento de enumerar servicios del servidor."
    elif "vpn-tunnel established" in row["raw"]:
        tipo = "Acceso VPN"
        interpretation = "Se establece un túnel VPN y se asigna 10.8.0.37."
    elif "45.77.132.90" in row["raw"]:
        tipo = "Tráfico de salida"
        interpretation = "Comunicación HTTP del servidor de reportes hacia infraestructura externa."
    else:
        tipo = "Alerta de red"
        interpretation = "El firewall registra un cambio o umbral de tráfico."
    timeline.append({
        "timestamp": row["timestamp"],
        "fuente": "firewall.log",
        "tipo": tipo,
        "hecho_observado": row["raw"],
        "interpretacion": interpretation,
        "certeza": "Alta (bitácora)",
    })

# Windows events
for _, row in df_events.iterrows():
    timeline.append({
        "timestamp": row["timestamp"],
        "fuente": "WIN_SERVICIOS_events.log",
        "tipo": f"Windows Event ID {row['event_id']}",
        "hecho_observado": row["message"],
        "interpretacion": "Evento objetivo del servidor; debe correlacionarse con red y script.",
        "certeza": "Alta (evento registrado)",
    })

df_timeline = pd.DataFrame(timeline).dropna(subset=["timestamp"]).sort_values("timestamp")
display(df_timeline)
df_timeline.to_csv(RESULTADOS / "linea_tiempo_forense.csv", index=False)


## 13. Resumen de hallazgos sustentados

Complete o ajuste estas conclusiones después de revisar todas las tablas:

1. Se recibió una solicitud de un nuevo JWT desde un dominio que debe compararse con el dominio legítimo del socio.
2. Una conexión VPN recibió la dirección `10.8.0.37`.
3. Desde esa dirección se probaron diversos puertos contra `10.10.4.10`; el puerto 445 fue permitido.
4. Windows registró una solicitud SMB1, un reinicio por `bugcheck`, la creación de `puravida.js` y su ejecución con `wscript.exe`.
5. El script referencia cuatro CSV, utiliza HTTP POST hacia `/collect` y contiene una rutina de autoeliminación.
6. Windows registró acceso a los cuatro archivos y una conexión de `wscript.exe` hacia `45.77.132.90:80`.
7. El firewall y el PCAP deben usarse conjuntamente para confirmar la comunicación de salida y describir su duración.
8. La evidencia permite describir una exfiltración probable o confirmada según el contenido reconstruido en el PCAP; no debe atribuirse un CVE específico sin evidencia adicional.


## 14. Exportación

Los resultados se guardan en `resultados/`. En Binder, descárguelos antes de cerrar la sesión porque el entorno es temporal.


In [ ]:
print("Archivos generados:")
for path in sorted(RESULTADOS.glob("*.csv")):
    print("-", path.relative_to(BASE))
